# WBC White Blood Cell Classification
## Full Pipeline — Data Exploration + Preprocessing + Training + Submission

In [ ]:
!pip install timm albumentations --quiet

## 1. Imports

In [ ]:
import os, random
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from scipy.optimize import minimize

import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm

print("PyTorch:", torch.__version__)
print("Timm:   ", timm.__version__)
print("GPU:    ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Configuration

In [ ]:
CONFIG = {
    "img_size"   : 384,
    "batch_size" : 32,
    "epochs"     : 60,
    "lr"         : 2e-4,
    "num_classes": 13,
    "seed"       : 42,
    "num_workers": 4,
}

base_path       = "/home/aellefi-25/IMA205-challenge"
train_path      = os.path.join(base_path, "train_metadata.csv")
test_path       = os.path.join(base_path, "test_metadata.csv")
sample_sub_path = os.path.join(base_path, "sample_submission.csv")
train_images    = os.path.join(base_path, "train")
test_images     = os.path.join(base_path, "test")

CACHE_DIR       = "/home/aellefi-25/wbc_cache"
BEST_MODEL_PATH = "/home/aellefi-25/best_model.pth"
SUBMISSION_PATH = "/home/aellefi-25/submission.csv"

RARE_CLASSES   = {"PLY", "PC", "PMY"}
SEVERE_CLASSES = {"MMY", "VLY", "BNE", "BA", "MY"}
ALL_RARE       = RARE_CLASSES | SEVERE_CLASSES
CONFUSED_PAIRS = [
    ("BNE", "MMY"), ("MMY", "MY"), ("LY", "VLY"),
    ("PLY", "LY"),  ("PC", "BL"),  ("PMY", "MY"),
]

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CONFIG["seed"])
print("Config ready.")

## 3. Data Loading

In [ ]:
train_df   = pd.read_csv(train_path)
test_df    = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_sub_path)

classes  = sorted(train_df["label"].unique())
label2id = {label: i for i, label in enumerate(classes)}
id2label = {i: label for label, i in label2id.items()}
train_df["label_id"] = train_df["label"].map(label2id)

RARE_IDS = {label2id[c] for c in ALL_RARE if c in label2id}

print("Train shape:", train_df.shape)
print("Test shape: ", test_df.shape)
print("\nLabel mapping:")
for label, idx in label2id.items():
    count = (train_df["label"] == label).sum()
    flag  = " *** CRITICAL" if label in RARE_CLASSES else \
            " ** SEVERE"   if label in SEVERE_CLASSES else ""
    print(f"  {idx:2d} — {label:4s} — {count:5d} images{flag}")

## 4. Data Exploration

In [ ]:
# Class distribution bar chart
counts = train_df["label"].value_counts()
colors = ["red"       if c in RARE_CLASSES   else
          "orange"    if c in SEVERE_CLASSES else
          "steelblue" for c in counts.index]

plt.figure(figsize=(13, 4))
bars = plt.bar(counts.index, counts.values, color=colors)
for bar, val in zip(bars, counts.values):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 30, str(val),
             ha="center", fontsize=8)
plt.title("Class Distribution — Red=Critical, Orange=Severe, Blue=Normal")
plt.ylabel("Count")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig("/home/aellefi-25/class_distribution.png", dpi=150)
plt.show()
print(f"\nTotal train images: {len(train_df)}")
print(f"Total test images:  {len(test_df)}")

In [ ]:
# Show sample images for each class
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.flatten()
for i, label in enumerate(classes):
    sample = train_df[train_df["label"] == label].iloc[0]
    img    = np.array(Image.open(
        os.path.join(train_images, sample["ID"])).convert("RGB"))
    axes[i].imshow(img)
    flag = " (CRITICAL)" if label in RARE_CLASSES else \
           " (SEVERE)"   if label in SEVERE_CLASSES else ""
    axes[i].set_title(f"{label}{flag}", fontsize=10)
    axes[i].axis("off")
for j in range(i+1, len(axes)):
    axes[j].axis("off")
plt.suptitle("Sample Images per Class", fontsize=14)
plt.tight_layout()
plt.savefig("/home/aellefi-25/sample_images.png", dpi=150)
plt.show()

## 5. Preprocessing Functions

Three-stage pipeline: denoising → stain normalisation → CLAHE contrast enhancement.

In [ ]:
def get_noise_level(image):
    gray    = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    if lap_var > 10000: return "extreme"
    if lap_var > 2000:  return "medium"
    return "clean"

def denoise_image(image):
    level = get_noise_level(image)
    if level == "extreme":
        return cv2.fastNlMeansDenoisingColored(
            image, None, h=40, hColor=40,
            templateWindowSize=7, searchWindowSize=21)
    if level == "medium":
        return cv2.fastNlMeansDenoisingColored(
            image, None, h=10, hColor=10,
            templateWindowSize=7, searchWindowSize=21)
    return image

def normalize_stain(image):
    image_float = image.astype(np.float32) / 255.0
    lab         = cv2.cvtColor(image_float, cv2.COLOR_RGB2LAB)
    target_mean = np.array([70.0,  5.0, -10.0])
    target_std  = np.array([15.0,  8.0,   8.0])
    for i in range(3):
        ch = lab[:, :, i]
        ch = (ch - ch.mean()) / (ch.std() + 1e-6)
        lab[:, :, i] = ch * target_std[i] + target_mean[i]
    return np.clip(
        cv2.cvtColor(lab, cv2.COLOR_LAB2RGB) * 255, 0, 255
    ).astype(np.uint8)

def apply_clahe(image):
    lab     = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l       = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)

def preprocess_image(image):
    image = denoise_image(image)
    image = normalize_stain(image)
    image = apply_clahe(image)
    return image

print("Preprocessing functions defined.")

### 5.1 Preprocessing Visualisation

In [ ]:
def show_preprocessing_pipeline(class_name, n=2):
    df_cls = train_df[train_df["label"] == class_name].head(n)
    fig, axes = plt.subplots(n, 4, figsize=(16, 4*n))
    if n == 1: axes = [axes]
    for i, (_, row) in enumerate(df_cls.iterrows()):
        img        = np.array(Image.open(os.path.join(train_images, row["ID"])).convert("RGB"))
        denoised   = denoise_image(img)
        normalized = normalize_stain(denoised)
        final      = apply_clahe(normalized)
        for ax, im, title in zip(axes[i],
            [img, denoised, normalized, final],
            ["Original", "Denoised", "Stain Norm", "CLAHE"]):
            ax.imshow(im); ax.set_title(title); ax.axis("off")
    plt.suptitle(f"Preprocessing pipeline — {class_name}", fontsize=13)
    plt.tight_layout()
    plt.savefig(f"/home/aellefi-25/preprocess_{class_name}.png", dpi=150)
    plt.show()

# Show for most affected classes
for cls in ["BNE", "VLY", "LY", "MMY"]:
    show_preprocessing_pipeline(cls)

## 6. Train / Validation Split

In [ ]:
train_data, val_data = train_test_split(
    train_df, test_size=0.2,
    stratify=train_df["label"],
    random_state=CONFIG["seed"]
)
train_data = train_data.reset_index(drop=True)
val_data   = val_data.reset_index(drop=True)

print(f"Train: {len(train_data)} | Val: {len(val_data)}")
print("\nVal class distribution:")
for label, idx in label2id.items():
    print(f"  {label:4s} — {(val_data['label']==label).sum():4d}")

In [ ]:
class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(CONFIG["num_classes"]),
    y=train_data["label"].map(label2id).values
)
class_weights = torch.tensor(class_weights_np, dtype=torch.float).cuda()

print("Class weights:")
for label, idx in label2id.items():
    print(f"  {label:4s} — {class_weights[idx]:.4f}")

## 7. Data Augmentation

In [ ]:
train_transforms = A.Compose([
    A.Resize(CONFIG["img_size"], CONFIG["img_size"]),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=45, p=0.8),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.RGBShift(r_shift_limit=15, g_shift_limit=15, b_shift_limit=15, p=0.4),
    A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.5),
    A.RandomResizedCrop(
        size=(CONFIG["img_size"], CONFIG["img_size"]),
        scale=(0.8, 1.0), ratio=(0.9, 1.1), p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transforms = A.Compose([
    A.Resize(CONFIG["img_size"], CONFIG["img_size"]),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

tta_transforms = A.Compose([
    A.Resize(CONFIG["img_size"], CONFIG["img_size"]),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=20, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

print("Transforms defined.")

## 8. Dataset & DataLoaders

In [ ]:
class WBCDataset(Dataset):
    def __init__(self, df, image_dir, transforms=None, has_labels=True):
        self.df         = df.reset_index(drop=True)
        self.image_dir  = image_dir
        self.transforms = transforms
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row        = self.df.iloc[idx]
        cache_path = os.path.join(CACHE_DIR, row["ID"])
        if os.path.exists(cache_path):
            image = np.array(Image.open(cache_path).convert("RGB"))
        else:
            image = np.array(Image.open(
                os.path.join(self.image_dir, row["ID"])).convert("RGB"))
            image = preprocess_image(image)
            os.makedirs(CACHE_DIR, exist_ok=True)
            Image.fromarray(image).save(cache_path)
        if self.transforms:
            image = self.transforms(image=image)["image"]
        if self.has_labels:
            return image, label2id[row["label"]]
        return image

In [ ]:
BOOST = {
    "PLY": 8.0,
    "PC" : 5.0,
    "PMY": 4.0,
    "MMY": 3.0,
    "BNE": 3.0,
    "MY" : 3.0,
    "VLY": 2.5,
    "BA" : 2.0,
}

sample_weights = np.array([
    1.0 / np.sqrt(max((train_data["label"] == label).sum(), 1))
    * BOOST.get(label, 1.0)
    for label in train_data["label"].values
])
sample_weights = torch.tensor(sample_weights, dtype=torch.float)
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_dataset = WBCDataset(train_data, train_images, train_transforms)
val_dataset   = WBCDataset(val_data,   train_images, val_transforms)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    sampler=sampler,
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
    persistent_workers=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
    persistent_workers=True,
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## 9. Loss Functions

In [ ]:
class FocalLossSmooth(nn.Module):
    """Focal loss with label smoothing to prevent overconfidence on majority classes."""
    def __init__(self, weight=None, gamma=1.5, smoothing=0.1):
        super().__init__()
        self.weight    = weight
        self.gamma     = gamma
        self.smoothing = smoothing

    def forward(self, inputs, targets):
        n = inputs.size(1)
        with torch.no_grad():
            smooth = torch.full_like(inputs, self.smoothing / (n - 1))
            smooth.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        log_p = F.log_softmax(inputs, dim=1)
        ce    = -(smooth * log_p).sum(dim=1)
        w     = self.weight[targets] if self.weight is not None else torch.ones_like(ce)
        pt    = torch.exp(-ce)
        return (w * (1 - pt) ** self.gamma * ce).mean()


class SupConLoss(nn.Module):
    """Supervised contrastive loss to pull same-class embeddings together."""
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        features = F.normalize(features, dim=1)
        sim      = torch.matmul(features, features.T) / self.temperature
        sim      = sim - sim.max(dim=1, keepdim=True)[0].detach()
        labels   = labels.view(-1, 1)
        mask     = torch.eq(labels, labels.T).float().to(features.device)
        eye      = torch.eye(len(labels)).to(features.device)
        mask     = mask - eye
        exp_sim  = torch.exp(sim) * (1 - eye)
        log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)
        loss     = -(mask * log_prob).sum(dim=1) / (mask.sum(dim=1) + 1e-8)
        return loss.mean()


focal_loss  = FocalLossSmooth(weight=class_weights, gamma=1.5, smoothing=0.1)
supcon_loss = SupConLoss(temperature=0.07)
print("Losses defined.")

## 10. Training Augmentation Helpers

In [ ]:
def cutmix_batch(images, labels, alpha=1.0):
    """CutMix — only applied when batch contains no rare class images."""
    lam   = np.random.beta(alpha, alpha)
    idx   = torch.randperm(images.size(0)).to(images.device)
    W, H  = images.size(2), images.size(3)
    cut_w = int(W * np.sqrt(1 - lam))
    cut_h = int(H * np.sqrt(1 - lam))
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1 = max(0, cx - cut_w // 2); x2 = min(W, cx + cut_w // 2)
    y1 = max(0, cy - cut_h // 2); y2 = min(H, cy + cut_h // 2)
    mixed = images.clone()
    mixed[:, :, x1:x2, y1:y2] = images[idx, :, x1:x2, y1:y2]
    lam   = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    return mixed, labels, labels[idx], lam


def rare_class_mixup(images, labels, alpha=0.4):
    """Intra-class MixUp for rare classes only — never crosses class boundaries."""
    mixed = images.clone()
    for i in range(len(labels)):
        if labels[i].item() not in RARE_IDS:
            continue
        same = (labels == labels[i]).nonzero(as_tuple=True)[0]
        same = same[same != i]
        if len(same) == 0:
            continue
        j         = same[torch.randint(len(same), (1,)).item()].item()
        lam       = np.random.beta(alpha, alpha)
        mixed[i]  = lam * images[i] + (1 - lam) * images[j]
    return mixed

print("Augmentation helpers defined.")

## 11. Training Loop

In [ ]:
def train_one_epoch(model, loader, focal_loss, supcon_loss,
                    optimizer, scheduler,
                    supcon_weight=0.03, cutmix_prob=0.5):
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []

    for images, labels in tqdm(loader, desc="Train", leave=False):
        images, labels = images.cuda(), labels.cuda()
        optimizer.zero_grad()

        images   = rare_class_mixup(images, labels)
        has_rare = any(l.item() in RARE_IDS for l in labels)
        use_cutmix = (not has_rare) and (random.random() < cutmix_prob)

        if use_cutmix:
            images, labels_a, labels_b, lam = cutmix_batch(images, labels)
            logits, feats = model(images, return_features=True)
            loss = (lam * focal_loss(logits, labels_a) +
                    (1 - lam) * focal_loss(logits, labels_b))
        else:
            logits, feats = model(images, return_features=True)
            loss          = focal_loss(logits, labels)
            loss          = loss + supcon_weight * supcon_loss(feats, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return total_loss / len(loader), f1


def validate(model, loader, focal_loss, return_probs=False):
    model.eval()
    total_loss, all_preds, all_labels, all_probs = 0.0, [], [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Val", leave=False):
            images, labels = images.cuda(), labels.cuda()
            logits = model(images)
            loss   = focal_loss(logits, labels)
            probs  = torch.softmax(logits, dim=1).cpu().numpy()
            total_loss += loss.item()
            all_probs.append(probs)
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    f1        = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    all_probs = np.concatenate(all_probs, axis=0)
    if return_probs:
        return total_loss / len(loader), f1, np.array(all_labels), np.array(all_preds), all_probs
    return total_loss / len(loader), f1, np.array(all_labels), np.array(all_preds)


def run_training(model, train_loader, val_loader,
                 focal_loss, supcon_loss,
                 optimizer, scheduler,
                 n_epochs, start_epoch,
                 best_f1, best_model_path, patience=8):

    history    = {"train_loss": [], "train_f1": [], "val_loss": [], "val_f1": []}
    no_improve = 0

    for epoch in range(start_epoch, n_epochs):
        print(f"\nEpoch {epoch+1}/{n_epochs} " + "-"*30, flush=True)
        train_loss, train_f1 = train_one_epoch(
            model, train_loader, focal_loss, supcon_loss, optimizer, scheduler)
        val_loss, val_f1, _, _ = validate(model, val_loader, focal_loss)

        lr = optimizer.param_groups[0]["lr"]
        history["train_loss"].append(train_loss)
        history["train_f1"].append(train_f1)
        history["val_loss"].append(val_loss)
        history["val_f1"].append(val_f1)
        print(f"  Train  loss={train_loss:.4f}  f1={train_f1:.4f}", flush=True)
        print(f"  Val    loss={val_loss:.4f}  f1={val_f1:.4f}  lr={lr:.2e}", flush=True)

        if val_f1 > best_f1:
            best_f1    = val_f1
            no_improve = 0
            torch.save(model.state_dict(), best_model_path)
            print(f"  *** Best saved — Val F1: {best_f1:.4f}", flush=True)
        else:
            no_improve += 1
            print(f"  No improvement ({no_improve}/{patience})", flush=True)
            if no_improve >= patience:
                print(f"  Early stopping at epoch {epoch+1}", flush=True)
                break

    print(f"\nDone. Best Val F1: {best_f1:.4f}", flush=True)
    return best_f1, history

print("Training functions defined.")

## 12. Model — ConvNeXt-Base

In [ ]:
class WBCModel(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.4):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=True,
            num_classes=0, global_pool="avg")
        in_features = self.backbone.num_features

        self.projector = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )
        self.head = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Dropout(dropout),
            nn.Linear(in_features, 768),
            nn.GELU(),
            nn.BatchNorm1d(768),
            nn.Dropout(dropout * 0.5),
            nn.Linear(768, 256),
            nn.GELU(),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout * 0.25),
            nn.Linear(256, num_classes)
        )

    def forward(self, x, return_features=False):
        features = self.backbone(x)
        logits   = self.head(features)
        if return_features:
            return logits, self.projector(features)
        return logits


model   = WBCModel("convnext_base.fb_in22k_ft_in1k", CONFIG["num_classes"]).cuda()
total_p = sum(p.numel() for p in model.parameters()) / 1e6
print(f"ConvNeXt-Base — {total_p:.1f}M params")

## 13. Two-Phase Training

In [ ]:
# Phase 1 — freeze backbone, train head only
for param in model.backbone.parameters():
    param.requires_grad = False
print("Phase 1 — backbone frozen, training head only...")

optimizer_p1 = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG["lr"], weight_decay=1e-4)
scheduler_p1 = optim.lr_scheduler.OneCycleLR(
    optimizer_p1, max_lr=CONFIG["lr"],
    steps_per_epoch=len(train_loader),
    epochs=5, pct_start=0.2,
    anneal_strategy="cos",
    div_factor=25, final_div_factor=1e4)

best_f1, history_p1 = run_training(
    model, train_loader, val_loader,
    focal_loss, supcon_loss,
    optimizer_p1, scheduler_p1,
    n_epochs=5, start_epoch=0,
    best_f1=0.0, best_model_path=BEST_MODEL_PATH, patience=5)

# Phase 2 — unfreeze all, fine-tune
for param in model.backbone.parameters():
    param.requires_grad = True
print("\nPhase 2 — all layers unfrozen, fine-tuning...")

optimizer_p2 = optim.AdamW(
    model.parameters(), lr=CONFIG["lr"] / 5, weight_decay=1e-4)
scheduler_p2 = optim.lr_scheduler.OneCycleLR(
    optimizer_p2, max_lr=CONFIG["lr"] / 5,
    steps_per_epoch=len(train_loader),
    epochs=CONFIG["epochs"],
    pct_start=0.1, anneal_strategy="cos",
    div_factor=25, final_div_factor=1e4)

best_f1, history_p2 = run_training(
    model, train_loader, val_loader,
    focal_loss, supcon_loss,
    optimizer_p2, scheduler_p2,
    n_epochs=CONFIG["epochs"], start_epoch=0,
    best_f1=best_f1, best_model_path=BEST_MODEL_PATH, patience=12)

history = {k: history_p1[k] + history_p2[k] for k in history_p1}
print(f"\nFinal best Val F1: {best_f1:.4f}")

## 14. Results & Visualisation

In [ ]:
def plot_history(history, title):
    epochs = range(1, len(history["val_f1"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(epochs, history["train_loss"], label="Train", marker="o", markersize=3)
    ax1.plot(epochs, history["val_loss"],   label="Val",   marker="o", markersize=3)
    ax1.set_title(f"{title} — Loss"); ax1.set_xlabel("Epoch")
    ax1.legend(); ax1.grid(True, alpha=0.3)
    ax2.plot(epochs, history["train_f1"], label="Train", marker="o", markersize=3)
    ax2.plot(epochs, history["val_f1"],   label="Val",   marker="o", markersize=3)
    ax2.axhline(y=max(history["val_f1"]), color="red", linestyle="--",
                alpha=0.5, label=f"Best={max(history['val_f1']):.4f}")
    ax2.set_title(f"{title} — Macro F1"); ax2.set_xlabel("Epoch")
    ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"/home/aellefi-25/{title.replace(' ','_')}_curves.png", dpi=150)
    plt.show()

plot_history(history, "ConvNeXt-Base")

# Load best checkpoint and get val predictions
model.load_state_dict(torch.load(BEST_MODEL_PATH))
_, _, val_labels, val_preds, val_probs = validate(
    model, val_loader, focal_loss, return_probs=True)

# Per-class F1
f1_per_class = f1_score(val_labels, val_preds, average=None, zero_division=0)
plt.figure(figsize=(13, 5))
bars = plt.bar(classes, f1_per_class, color=[
    "red"       if c in RARE_CLASSES   else
    "orange"    if c in SEVERE_CLASSES else
    "steelblue" for c in classes])
for bar, val in zip(bars, f1_per_class):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.01, f"{val:.2f}",
             ha="center", fontsize=9)
plt.title(f"Per-class F1 (Best val={best_f1:.4f})")
plt.ylabel("F1 Score"); plt.ylim(0, 1.15)
plt.xticks(rotation=30); plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("/home/aellefi-25/per_class_f1.png", dpi=150)
plt.show()

print(classification_report(val_labels, val_preds,
                             target_names=classes, zero_division=0))

In [ ]:
cm      = confusion_matrix(val_labels, val_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
plt.figure(figsize=(11, 9))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=classes, yticklabels=classes)
plt.title("Confusion Matrix (normalised)")
plt.ylabel("True"); plt.xlabel("Predicted")
plt.tight_layout()
plt.savefig("/home/aellefi-25/confusion_matrix.png", dpi=150)
plt.show()

## 15. Post-processing — Temperature Scaling + Threshold Optimisation

In [ ]:
def find_temperature(val_probs, val_labels):
    """Learn a scalar temperature T that maximises macro F1 on val set."""
    def neg_f1(log_t):
        T      = float(np.exp(log_t[0]))
        scaled = val_probs ** (1.0 / T)
        scaled = scaled / scaled.sum(axis=1, keepdims=True)
        return -f1_score(val_labels, scaled.argmax(axis=1),
                         average="macro", zero_division=0)
    result = minimize(neg_f1, x0=[0.0], method="Nelder-Mead",
                      options={"maxiter": 5000})
    T      = float(np.exp(result.x[0]))
    scaled = val_probs ** (1.0 / T)
    scaled = scaled / scaled.sum(axis=1, keepdims=True)
    cal_f1 = f1_score(val_labels, scaled.argmax(axis=1),
                      average="macro", zero_division=0)
    print(f"Best temperature T = {T:.3f}  →  val F1: {cal_f1:.4f}")
    return T


def optimize_thresholds(probs, labels, n_classes=13):
    """Learn per-class scaling factors that maximise macro F1 on val set."""
    def neg_f1(t):
        scaled = probs / np.clip(t, 1e-3, None)
        return -f1_score(labels, scaled.argmax(axis=1),
                         average="macro", zero_division=0)
    result = minimize(neg_f1, x0=np.ones(n_classes), method="Nelder-Mead",
                      options={"maxiter": 20000, "xatol": 1e-5, "fatol": 1e-5})
    t      = np.clip(result.x, 1e-3, None)
    adj_f1 = f1_score(labels, (probs / t).argmax(axis=1),
                      average="macro", zero_division=0)
    print(f"Raw val F1:            {f1_score(labels, probs.argmax(axis=1), average='macro', zero_division=0):.4f}")
    print(f"Threshold-adjusted F1: {adj_f1:.4f}")
    print("\nPer-class thresholds:")
    for label, idx in label2id.items():
        print(f"  {label:4s}  t={t[idx]:.4f}")
    return t


best_T          = find_temperature(val_probs, val_labels)
best_thresholds = optimize_thresholds(val_probs, val_labels, CONFIG["num_classes"])

## 16. TTA Inference & Submission

In [ ]:
def predict_tta(model, test_df, image_dir, n_tta=15):
    model.eval()
    all_probs = []
    for i in range(n_tta):
        aug = val_transforms if i == 0 else tta_transforms
        ds  = WBCDataset(test_df, image_dir, aug, has_labels=False)
        dl  = DataLoader(ds, batch_size=CONFIG["batch_size"],
                         shuffle=False, num_workers=CONFIG["num_workers"],
                         pin_memory=True)
        probs = []
        with torch.no_grad():
            for images in tqdm(dl, desc=f"TTA {i+1}/{n_tta}", leave=False):
                out = model(images.cuda())
                probs.append(torch.softmax(out, dim=1).cpu().numpy())
        all_probs.append(np.concatenate(probs, axis=0))
    return np.mean(all_probs, axis=0)

model.load_state_dict(torch.load(BEST_MODEL_PATH))
print("Running TTA inference (15 passes)...")
probs_test = predict_tta(model, test_df, test_images, n_tta=15)

# Temperature scaling
probs_cal = probs_test ** (1.0 / best_T)
probs_cal = probs_cal / probs_cal.sum(axis=1, keepdims=True)

# Per-class threshold scaling
probs_adj = probs_cal / np.clip(best_thresholds, 1e-3, None)

pred_ids    = probs_adj.argmax(axis=1)
pred_labels = [id2label[i] for i in pred_ids]

print("\nPrediction distribution:")
print(pd.Series(pred_labels).value_counts())

sub_out = pd.DataFrame({"ID": test_df["ID"].values, "label": pred_labels})
sub_out.to_csv(SUBMISSION_PATH, index=False)

assert len(sub_out) == len(test_df)
assert set(sub_out["label"].unique()).issubset(set(classes))
print(f"\nSubmission saved — {len(sub_out)} rows → {SUBMISSION_PATH}")
print(sub_out.head())